In [8]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import time
import gget
import anndata as an
import scanpy as sc
import scanpy.external as sce
import h5py
import scipy
from scipy.stats import pearsonr
from scipy.spatial.distance import cdist
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
import re
import matplotlib.colors as mcolors
import networkx as nx
import itertools
from scipy.linalg import eigh

from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from collections import defaultdict

sc.settings.verbosity = 2

In [2]:
%%time
path = "/nfs/turbo/umms-indikar/shared/projects/hybrid_reprogramming/anndata/processed_all_groups.h5ad"
adata = sc.read_h5ad(path)
adata.uns['phase_colors'] = ['#ad3803', '#56b567', 'gold']
sc.logging.print_memory_usage()
adata

Memory usage: current 2.38 GB, difference +2.38 GB
CPU times: user 243 ms, sys: 1.14 s, total: 1.38 s
Wall time: 4.35 s


/nfs/turbo/umms-indikar/Jillian/conda-envs/rapids/lib/python3.13/site-packages/anndata/logging.py:57: FutureWarning: The specified parameters ('newline',) are no longer positional. Please specify them like `newline=False`
  print(format_memory_usage(get_memory_usage(), msg, newline))


AnnData object with n_obs × n_vars = 15950 × 25042
    obs: 'MYOD-fb_counts', 'PRRX1-fb_counts', 'PRRX1_MYOD-fb_counts', 'assigned_condition', 'total_fb_counts', 'condition_counts_rate', 'G1-fb_counts', 'G2M-fb_counts', 'S-fb_counts', 'dataset', 'total_reads', 'total_genes', 'pooled_condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'outlier', 'mt_outlier', 'S_score', 'G2M_score', 'phase', 'leiden', 'cluster_str', 'leiden_split'
    var: 'gene_id', 'gene_type', 'Chromosome', 'Start', 'End', 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'filter_pass', 'highly_variable', 'highly_variable_rank', 'm

# Shannon

In [3]:
### Core function

def shannon_entropy(probs, base=2):
    """
    probs: array-like of probabilities (must sum to 1)
    """
    
    probs = np.asarray(probs)
    probs = probs[probs > 0] # avoid log(0)
    
    return -np.sum(probs * np.log(probs)) / np.log(base)

## per condition entropy

In [4]:
def condition_entropy_from_labels(adata, condition_key, label_key):
    results = {}
    
    for cond in adata.obs[condition_key].unique():
        obs_sub = adata.obs[adata.obs[condition_key] == cond]
        
        counts = obs_sub[label_key].value_counts(normalize=True)
        entropy = shannon_entropy(counts.values)
        
        results[cond] = entropy
        
    return pd.Series(results, name=f"Shannon_entropy_{label_key}")

In [5]:
# entropy of cluster distributions per condition

cluster_entropy = condition_entropy_from_labels(
    adata,
    condition_key='pooled_condition',
    label_key='leiden_split',
)

print(cluster_entropy) # lower entropy here means there's a dominant attractor state, but each condition was fixed at 3 clusters

siPRRX1/mmMYOD1    1.372537
mmMYOD1            1.584218
siPRRX1            1.531280
Control            0.592725
Name: Shannon_entropy_leiden_split, dtype: float64


In [6]:
# entropy of cell cycle phase distributions per condition

phase_entropy = condition_entropy_from_labels(
    adata,
    condition_key='pooled_condition',
    label_key='phase',
)

print(phase_entropy) # high entropy = loss of cell cycle coordination? low entropy = structured progression ?

siPRRX1/mmMYOD1    1.453027
mmMYOD1            1.234162
siPRRX1            1.464718
Control            1.168219
Name: Shannon_entropy_phase, dtype: float64


In [7]:
# expression entropy per cell

# def per_cell_expression_entropy(adata, layer='log_norm', base=2):
#     X = adata.layers[layer]
    
#     if sp.issparse(X):
#         X = X.toarray()
        
#     # ensure positivity
#     X = np.maximum(X, 0)
    
#     # normalize per cell
#     row_sums = X.sum(axis=1, keepdims=True)
#     row_sums[row_sums == 0] = 1
#     P = X / row_sums
    
#     entropies = np.array([
#         shannon_entropy(row, base=base) for row in P
#     ])
    
#     return entropies


def per_cell_expression_entropy(adata, layer='log_norm', base=2):
    X = adata.layers[layer]
    
    # Work with sparse matrices directly to save memory
    if sp.issparse(X):
        # We need the sum per row (cell)
        row_sums = np.array(X.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1
        
        # Manually compute -sum(p * log(p))
        # We only care about non-zero entries for entropy
        data = X.data
        rows, cols = X.nonzero()
        
        # Calculate p for each non-zero entry
        p = data / row_sums[rows]
        
        # Shannon entropy component: -p * log(p)
        log_p = np.log(p) / np.log(base)
        vals = -p * log_p
        
        # Aggregate back to cell totals
        entropies = np.zeros(X.shape[0])
        np.add.at(entropies, rows, vals)
    else:
        # Vectorized dense version
        X = np.maximum(X, 0)
        row_sums = X.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        P = X / row_sums
        
        # Use np.errstate to ignore log(0) warnings
        with np.errstate(divide='ignore', invalid='ignore'):
            vals = -P * (np.log(P) / np.log(base))
        vals[np.isnan(vals)] = 0
        entropies = vals.sum(axis=1)
        
    return entropies



adata.obs['expression_entropy'] = per_cell_expression_entropy(
    adata,
    layer='log_norm',
)

expr_entropy_by_condition = (
    adata.obs
    .groupby('pooled_condition')['expression_entropy']
    .mean()
)

print(expr_entropy_by_condition)

pooled_condition
Control            11.559952
mmMYOD1            11.662240
siPRRX1            11.752680
siPRRX1/mmMYOD1    11.693696
Name: expression_entropy, dtype: float64


/tmp/ipykernel_1320940/999024859.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('pooled_condition')['expression_entropy']


# Von Neumann

In [10]:
def get_expression_matrix(adata, condition, layer='log_norm'):
    idx = adata.obs['pooled_condition'] == condition
    X = adata[idx].layers[layer]
    
    if sp.issparse(X):
        X = X.toarray()
        
    return X


def correlation_adjacency(X, method='pearson'):
    # X = cells x genes
    
    X = X - X.mean(axis=0, keepdims=True)
    
    cov = np.cov(X, rowvar=False)
    
    std = np.sqrt(np.diag(cov))
    std[std == 0] = 1
    
    corr = cov / np.outer(std, std)
    
    # remove self loops
    np.fill_diagonal(corr, 0)
    
    return corr


def normalized_laplacian(A):
    A = np.abs(A)
    D = np.diag(A.sum(axis=1))
    
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D) + 1e-8))
    L = np.eye(A.shape[0]) - D_inv_sqrt @ A @ D_inv_sqrt
    
    return L



def von_neumann_entropy(L, base=2):
    # normalize to density matrix
    rho = L / np.trace(L)
    
    # eigenvalues
    
    eigvals = eigh(rho, eigvals_only=True)
    eigvals = eigvals[eigvals > 1e-12]
    
    entropy = -np.sum(eigvals * np.log(eigvals)) / np.log(base)
    
    return entropy


def graph_entropy_per_condition(adata, layer='log_norm'):
    results = {}
    
    for cond in adata.obs['pooled_condition'].unique():
        print(f'Processing {cond}...')
        
        X = get_expression_matrix(adata, cond, layer=layer)
        A = correlation_adjacency(X)
        L = normalized_laplacian(A)
        
        S = von_neumann_entropy(L)
        results[cond] = S
        
    return pd.Series(results, name='von_neumann_entropy')

In [36]:
def get_expression_matrix(adata, group_name, column, layer='log_norm'):
    """Subsets adata by a specific group in a specific column."""
    idx = adata.obs[column] == group_name
    X = adata[idx].layers[layer]
    
    if sp.issparse(X):
        X = X.toarray()
    return X

def correlation_adjacency(X, threshold=0.1):
    """Calculates correlation and handles zero-variance genes."""
    # 1. Identify genes that actually vary in this subset
    # This prevents division by zero (NaNs)
    gene_vars = np.var(X, axis=0)
    valid_genes = gene_vars > 0
    
    # Subset X to only include varying genes
    X_filt = X[:, valid_genes]
    
    if X_filt.shape[1] == 0:
        return np.zeros((0, 0))

    # 2. Calculate Pearson Correlation
    # np.corrcoef is more robust than manual cov/std division
    corr_filt = np.corrcoef(X_filt, rowvar=False)
    
    # 3. Put the filtered correlations back into a full-sized matrix 
    # (Optional: keeps the matrix size consistent with the gene list)
    num_genes = X.shape[1]
    corr = np.zeros((num_genes, num_genes))
    
    # Use meshgrid to map the filtered indices back to the full matrix
    ix = np.where(valid_genes)[0]
    ii, jj = np.meshgrid(ix, ix, indexing='ij')
    corr[ii, jj] = corr_filt
    
    # 4. Cleanup
    np.fill_diagonal(corr, 0)
    corr = np.nan_to_num(corr) # Safety net: convert NaNs to 0
    
    if threshold > 0:
        corr[np.abs(corr) < threshold] = 0
        
    return corr


def normalized_laplacian(A):
    """Computes the normalized laplacian of an adjacency matrix."""
    A_abs = np.abs(A)
    # Degree matrix: sum of absolute weights
    d = A_abs.sum(axis=1)
    
    # Handle isolated nodes (degree 0) to avoid division by zero
    d_inv_sqrt = np.zeros_like(d)
    nonzero = d > 1e-8
    d_inv_sqrt[nonzero] = 1.0 / np.sqrt(d[nonzero])
    
    D_inv_sqrt = np.diag(d_inv_sqrt)
    L = np.eye(A.shape[0]) - D_inv_sqrt @ A_abs @ D_inv_sqrt
    return L

def von_neumann_entropy(L, base=2):
    """Calculates VNE from the Laplacian eigenvalues."""
    # The trace of a normalized Laplacian is the number of non-isolated nodes
    tr = np.trace(L)
    if tr == 0: return 0
    
    rho = L / tr
    
    eigvals = eigh(rho, eigvals_only=True)
    # Filter for numerical stability and valid log domain
    eigvals = eigvals[eigvals > 1e-12]
    
    entropy = -np.sum(eigvals * np.log(eigvals)) / np.log(base)
    return entropy

def graph_entropy_by_group(adata, groupby, layer='log_norm', threshold=0.1):
    """
    Computes VNE for each group in adata.obs[groupby].
    groupby: e.g., 'pooled_condition' or 'leiden_split'
    """
    results = {}
    groups = adata.obs[groupby].unique()
    
    for group in groups:
        print(f'Processing {groupby}: {group}...')
        
        X = get_expression_matrix(adata, group, groupby, layer=layer)
        
        # Check if group is too small
        if X.shape[0] < 2:
            print(f"Skipping {group}: not enough cells.")
            continue
            
        A = correlation_adjacency(X, threshold=threshold)
        L = normalized_laplacian(A)
        S = von_neumann_entropy(L)
        
        results[group] = S
        
    return pd.Series(results, name=f'vne_by_{groupby}')

In [37]:
# subset for specific genes (otherwise would take forever to run/crash)
adata_hvg = adata[:, adata.var.highly_variable].copy()
print(adata_hvg.shape)

print(f"\nMax entropy: {np.log2(adata_hvg.shape[1])}\n")

cond_entropy = graph_entropy_by_group(adata_hvg, groupby='pooled_condition', threshold=0.1)
print(cond_entropy)

(15950, 2000)

Max entropy: 10.965784284662087

Processing pooled_condition: siPRRX1/mmMYOD1...
Processing pooled_condition: mmMYOD1...
Processing pooled_condition: siPRRX1...
Processing pooled_condition: Control...
siPRRX1/mmMYOD1    10.950216
mmMYOD1            10.952596
siPRRX1            10.915775
Control            10.871073
Name: vne_by_pooled_condition, dtype: float64


In [48]:
cluster_entropy = graph_entropy_by_group(adata_hvg, groupby='leiden_split', threshold=0.1)
print(cluster_entropy)

Processing leiden_split: H1...
Processing leiden_split: M1...
Processing leiden_split: P2...
Processing leiden_split: H2...
Processing leiden_split: M2...
Processing leiden_split: P1...
Processing leiden_split: P3...
Processing leiden_split: M3...
Processing leiden_split: H3...
Processing leiden_split: C1...
Processing leiden_split: C2...
Processing leiden_split: C3...
H1    10.941326
M1    10.958858
P2    10.927420
H2    10.955881
M2    10.954015
P1    10.940896
P3    10.952940
M3    10.960455
H3    10.961336
C1    10.879030
C2    10.947894
C3    10.964450
Name: vne_by_leiden_split, dtype: float64


In [16]:
# TFs
fpath = "../../resources/HumanTF_v_1.01.csv"

tdf = pd.read_csv(fpath)

tdf = tdf[['HGNC symbol', 'Is TF?']]
tdf = tdf.rename(columns={'HGNC symbol': 'gene_name', 'Is TF?': 'is_tf'})
tdf = tdf[tdf['is_tf'] == 'Yes']
tdf = tdf.drop_duplicates(subset='gene_name')
display(tdf.head())

tf_list = tdf['gene_name'].unique()

print(len(tf_list))

,gene_name,is_tf
0,TFAP2A,Yes
1,TFAP2B,Yes
2,TFAP2C,Yes
3,TFAP2D,Yes
4,TFAP2E,Yes


1639


In [38]:
adata_tf = adata[:, adata.var_names.isin(tf_list)].copy()
print(adata_tf.shape)

print(f"\nMax entropy: {np.log2(adata_tf.shape[1])}\n")

cond_entropy = graph_entropy_by_group(adata_tf, groupby='pooled_condition', threshold=0.1)
print(cond_entropy)

(15950, 1357)

Max entropy: 10.406205005418855

Processing pooled_condition: siPRRX1/mmMYOD1...
Processing pooled_condition: mmMYOD1...
Processing pooled_condition: siPRRX1...
Processing pooled_condition: Control...
siPRRX1/mmMYOD1    10.347544
mmMYOD1            10.323451
siPRRX1            10.317774
Control            10.354810
Name: vne_by_pooled_condition, dtype: float64


In [47]:
cluster_entropy = graph_entropy_by_group(adata_tf, groupby='leiden_split', threshold=0.1)
print(cluster_entropy)

Processing leiden_split: H1...
Processing leiden_split: M1...
Processing leiden_split: P2...
Processing leiden_split: H2...
Processing leiden_split: M2...
Processing leiden_split: P1...
Processing leiden_split: P3...
Processing leiden_split: M3...
Processing leiden_split: H3...
Processing leiden_split: C1...
Processing leiden_split: C2...
Processing leiden_split: C3...
H1    10.299536
M1    10.390364
P2    10.273127
H2    10.383624
M2    10.381735
P1    10.320285
P3    10.366736
M3    10.389501
H3    10.392581
C1    10.349591
C2    10.361539
C3    10.403519
Name: vne_by_leiden_split, dtype: float64


In [21]:
def load_pathway(fpath):
    result = []
    with open(fpath) as f:
        for line in f:
            split_line = [x for x in line.strip().split('\t') if x]  # Remove empty strings directly

            row = {'label': split_line[0]}
            for gene in split_line[1:]:
                row[gene] = 1

            result.append(row)

    df = pd.DataFrame(result)
    df = df.fillna(0.0).set_index('label').astype(bool).T  # Chained operations for clarity

    return df

mpath = "/nfs/turbo/umms-indikar/shared/projects/RECODE/marker_genes/PanglaoDB_Augmented_2021.txt"
pang = load_pathway(mpath)

cell_types = [
    'Fibroblasts',
    'Myocytes',
    'Myoblasts',
    'Myofibroblasts',
    'Pluripotent Stem Cells'
]

subset = pang[cell_types]

print("Marker genes per cell type:")
for ct in cell_types:
    print(f"{ct}: {subset[ct].sum()}")

panglao_markers = subset[subset.any(axis=1)]

print(f"\nTotal unique genes across selected types: {panglao_markers.shape[0]}")
panglao_markers.head()

Marker genes per cell type:
Fibroblasts: 232
Myocytes: 163
Myoblasts: 126
Myofibroblasts: 100
Pluripotent Stem Cells: 112

Total unique genes across selected types: 563


label,Fibroblasts,Myocytes,Myoblasts,Myofibroblasts,Pluripotent Stem Cells
RARRES2,True,False,False,False,False
CELA1,True,False,False,False,False
LUM,True,False,True,True,False
PRRX1,True,True,True,True,False
SCARA5,True,False,False,False,False


In [25]:
fib_genes = panglao_markers['Fibroblasts'] == True
fib_genes =  panglao_markers[fib_genes].index
fib_genes = [g for g in fib_genes if g in adata.var_names]
print(len(fib_genes))


myo_genes = (panglao_markers['Myocytes'] == True) | (panglao_markers['Myoblasts'] == True)
myo_genes =  panglao_markers[myo_genes].index
myo_genes = [g for g in myo_genes if g in adata.var_names]
print(len(myo_genes))

220
210


In [45]:
adata_tmp = adata[:, adata.var_names.isin(fib_genes)].copy()
print(adata_tmp.shape)

print(f"\nMax entropy: {np.log2(adata_tmp.shape[1])}\n")

cond_entropy = graph_entropy_by_group(adata_tmp, groupby='pooled_condition', threshold=0.1)
print(cond_entropy)

(15950, 220)

Max entropy: 7.78135971352466

Processing pooled_condition: siPRRX1/mmMYOD1...
Processing pooled_condition: mmMYOD1...
Processing pooled_condition: siPRRX1...
Processing pooled_condition: Control...
siPRRX1/mmMYOD1    7.738800
mmMYOD1            7.711681
siPRRX1            7.738851
Control            7.759461
Name: vne_by_pooled_condition, dtype: float64


In [46]:
cluster_entropy = graph_entropy_by_group(adata_tmp, groupby='leiden_split', threshold=0.1)
print(cluster_entropy)

Processing leiden_split: H1...
Processing leiden_split: M1...
Processing leiden_split: P2...
Processing leiden_split: H2...
Processing leiden_split: M2...
Processing leiden_split: P1...
Processing leiden_split: P3...
Processing leiden_split: M3...
Processing leiden_split: H3...
Processing leiden_split: C1...
Processing leiden_split: C2...
Processing leiden_split: C3...
H1    7.684887
M1    7.735839
P2    7.620469
H2    7.686715
M2    7.688653
P1    7.669050
P3    7.669103
M3    7.726239
H3    7.745624
C1    7.759928
C2    7.677191
C3    7.770443
Name: vne_by_leiden_split, dtype: float64


In [43]:
adata_tmp = adata[:, adata.var_names.isin(myo_genes)].copy()
print(adata_tmp.shape)

print(f"\nMax entropy: {np.log2(adata_tmp.shape[1])}\n")

cond_entropy = graph_entropy_by_group(adata_tmp, groupby='pooled_condition', threshold=0.1)
print(cond_entropy)

(15950, 210)

Max entropy: 7.714245517666122

Processing pooled_condition: siPRRX1/mmMYOD1...
Processing pooled_condition: mmMYOD1...
Processing pooled_condition: siPRRX1...
Processing pooled_condition: Control...
siPRRX1/mmMYOD1    7.617078
mmMYOD1            7.628804
siPRRX1            7.628037
Control            7.663712
Name: vne_by_pooled_condition, dtype: float64


In [44]:
cluster_entropy = graph_entropy_by_group(adata_tmp, groupby='leiden_split', threshold=0.1)
print(cluster_entropy)

Processing leiden_split: H1...
Processing leiden_split: M1...
Processing leiden_split: P2...
Processing leiden_split: H2...
Processing leiden_split: M2...
Processing leiden_split: P1...
Processing leiden_split: P3...
Processing leiden_split: M3...
Processing leiden_split: H3...
Processing leiden_split: C1...
Processing leiden_split: C2...
Processing leiden_split: C3...
H1    7.593990
M1    7.654683
P2    7.533437
H2    7.602220
M2    7.609643
P1    7.500913
P3    7.576391
M3    7.655049
H3    7.665270
C1    7.654604
C2    7.570444
C3    7.701644
Name: vne_by_leiden_split, dtype: float64


In [29]:
fpath = "../../resources/human_cell_cycle_genes.csv"

cdf = pd.read_csv(fpath)
go_cc_genes = cdf['gene_name'].unique()
print(len(go_cc_genes))


fpath = "../../resources/regev_lab_cell_cycle_genes.txt"
regev_genes = [x.strip() for x in open(fpath)]


cc_genes = list(set(go_cc_genes) | set(regev_genes))
print(f"N unique CC genes: {len(cc_genes)}")

142
N unique CC genes: 235


In [41]:
adata_tmp = adata[:, adata.var_names.isin(cc_genes)].copy()
print(adata_tmp.shape)

print(f"\nMax entropy: {np.log2(adata_tmp.shape[1])}\n")

cond_entropy = graph_entropy_by_group(adata_tmp, groupby='pooled_condition', threshold=0.1)
print(cond_entropy)

(15950, 210)

Max entropy: 7.714245517666122

Processing pooled_condition: siPRRX1/mmMYOD1...
Processing pooled_condition: mmMYOD1...
Processing pooled_condition: siPRRX1...
Processing pooled_condition: Control...
siPRRX1/mmMYOD1    7.693904
mmMYOD1            7.687478
siPRRX1            7.695677
Control            7.702252
Name: vne_by_pooled_condition, dtype: float64


In [42]:
cluster_entropy = graph_entropy_by_group(adata_tmp, groupby='leiden_split', threshold=0.1)
print(cluster_entropy)

Processing leiden_split: H1...
Processing leiden_split: M1...
Processing leiden_split: P2...
Processing leiden_split: H2...
Processing leiden_split: M2...
Processing leiden_split: P1...
Processing leiden_split: P3...
Processing leiden_split: M3...
Processing leiden_split: H3...
Processing leiden_split: C1...
Processing leiden_split: C2...
Processing leiden_split: C3...
H1    7.679131
M1    7.679746
P2    7.661735
H2    7.650702
M2    7.664267
P1    7.675986
P3    7.636768
M3    7.680877
H3    7.683755
C1    7.702436
C2    7.661380
C3    7.704145
Name: vne_by_leiden_split, dtype: float64
